# 🖼️ Conditional WGAN-GP — CIFAR-10 Class-Controlled Generation

**Dataset:** CIFAR-10 — 50,000 RGB images (32×32), 10 classes  
**Task:** Generate class-conditioned images given a label (airplane, car, bird...)

## From Vanilla GAN → WGAN-GP

| Property | Vanilla GAN | WGAN-GP |
|---|---|---|
| Loss | Binary Cross-Entropy | Wasserstein distance |
| Output | Probability (Sigmoid) | Raw score (no Sigmoid) |
| Lipschitz constraint | None | Gradient Penalty |
| Training stability | Often unstable | Much more stable |
| Critic updates per G step | 1 | 5 (or more) |

## Wasserstein Loss

**Critic** minimizes:
$$\mathcal{L}_C = \underbrace{\mathbb{E}[C(\tilde{x}, y)]}_{\text{fake score}} - \underbrace{\mathbb{E}[C(x, y)]}_{\text{real score}} + \underbrace{\lambda \cdot GP}_{\text{gradient penalty}}$$

**Generator** minimizes:
$$\mathcal{L}_G = -\mathbb{E}[C(G(z, y), y)]$$

## Gradient Penalty

Enforces 1-Lipschitz constraint on the critic:
$$GP = \lambda \cdot \mathbb{E}_{\hat{x}}\left[(\|\nabla_{\hat{x}} C(\hat{x}, y)\|_2 - 1)^2\right]$$

where $\hat{x} = \epsilon \cdot x + (1-\epsilon) \cdot \tilde{x}$, $\epsilon \sim U(0,1)$

## Label Conditioning

Both G and C receive the class label via `nn.Embedding`:
- **G**: label embed concatenated with noise z → controls what class to generate  
- **C**: label embed projected to spatial map → concatenated as extra image channel  

This forces C to evaluate whether image **matches** its label, not just looks realistic.

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

import yaml
import torch
import numpy as np
import matplotlib.pyplot as plt
import torchvision
from PIL import Image

from src.conditional_gan.generator     import ConditionalGenerator
from src.conditional_gan.discriminator import ConditionalDiscriminator
from src.conditional_gan.trainer       import ConditionalGANTrainer

CIFAR10_CLASSES = [
    "airplane", "automobile", "bird",  "cat",  "deer",
    "dog",      "frog",       "horse", "ship", "truck",
]

with open("../configs/conditional_gan_config.yaml", "r") as f:
    config = yaml.safe_load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Config loaded:")
for section, values in config.items():
    print(f"  {section}: {values}")
print(f"\nDevice: {device}")

In [ ]:
# Inspect model architectures before training
G = ConditionalGenerator(
    latent_dim     = config["model"]["latent_dim"],
    embedding_dim  = config["model"]["embedding_dim"],
    num_classes    = config["data"]["num_classes"],
    base_channels  = config["model"]["base_channels"],
    image_channels = config["data"]["image_channels"],
).to(device)

C = ConditionalDiscriminator(
    image_channels = config["data"]["image_channels"],
    num_classes    = config["data"]["num_classes"],
    embedding_dim  = config["model"]["embedding_dim"],
).to(device)

print("Generator architecture:")
print(G)
print(f"\nGenerator parameters : {G.count_parameters():,}")
print(f"Critic parameters    : {C.count_parameters():,}")

# Verify forward pass shapes
z_test      = torch.randn(4, config["model"]["latent_dim"]).to(device)
label_test  = torch.randint(0, 10, (4,)).to(device)
fake_test   = G(z_test, label_test)
score_test  = C(fake_test, label_test)

print(f"\nShape check:")
print(f"  z              : {z_test.shape}")
print(f"  labels         : {label_test.shape}")
print(f"  G(z, label)    : {fake_test.shape}")
print(f"  C(img, label)  : {score_test.shape}  ← raw score, no Sigmoid")

In [ ]:
# Train the model
# CPU: ~15 mins per epoch at batch_size=64, critic_iterations=2
# GPU: ~2 minutes per epoch, increase critic_iterations to 5
# Increase epochs to 50+ on GPU for quality results

trainer = ConditionalGANTrainer(config)
trainer.train()

history = trainer.get_history()
print(f"\nTraining complete.")
print(f"Final C Loss    : {history['c_losses'][-1]:.4f}")
print(f"Final G Loss    : {history['g_losses'][-1]:.4f}")
print(f"Final W Dist    : {history['w_distances'][-1]:.4f}")

In [ ]:
history = trainer.get_history()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, data, color, title, note in zip(
    axes,
    [history["c_losses"], history["g_losses"], history["w_distances"]],
    ["#E74C3C",           "#1A6BC4",           "#00C28B"],
    ["Critic Loss",        "Generator Loss",    "Wasserstein Distance"],
    ["Should stabilize",  "Should decrease",   "Rising = G improving"],
):
    ax.plot(data, color=color, linewidth=1.8)
    ax.set_xlabel("Epoch", fontsize=11)
    ax.set_title(title,    fontsize=12)
    ax.annotate(note, xy=(0.5, 0.04), xycoords="axes fraction",
                ha="center", fontsize=9, color="gray")
    ax.grid(alpha=0.3)

plt.suptitle("Conditional WGAN-GP — Training Curves (CIFAR-10)", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Generate 8 images per class and display labeled grid
# Each row = one class — shows conditioning quality directly

trainer.G.eval()
noise_list, label_list = [], []
for cls in range(10):
    noise_list.append(torch.randn(8, config["model"]["latent_dim"]))
    label_list.append(torch.full((8,), cls, dtype=torch.long))

noise  = torch.cat(noise_list).to(trainer.device)
labels = torch.cat(label_list).to(trainer.device)

with torch.no_grad():
    imgs = trainer.G(noise, labels)                        # (80, 3, 32, 32)

imgs_np = (imgs.cpu().clamp(-1, 1) + 1) / 2.0            # [-1,1] → [0,1]

fig, axes = plt.subplots(10, 8, figsize=(16, 22))
for cls in range(10):
    for col in range(8):
        img = imgs_np[cls * 8 + col].permute(1, 2, 0).numpy()
        axes[cls, col].imshow(img)
        axes[cls, col].axis("off")
    axes[cls, 0].set_ylabel(CIFAR10_CLASSES[cls], fontsize=11,
                            rotation=0, labelpad=55, va="center")

plt.suptitle("Conditional WGAN-GP — Generated Images per Class", fontsize=14)
plt.tight_layout()
os.makedirs("../assets/conditional_gan", exist_ok=True)
plt.savefig("../assets/conditional_gan/notebook_labeled_grid.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("Saved → assets/conditional_gan/notebook_labeled_grid.png")

In [ ]:
# Show generator quality progression across saved epochs
samples_dir = "../samples/conditional_gan"
epoch_files = sorted([
    f for f in os.listdir(samples_dir) if f.endswith(".png")
])

if not epoch_files:
    print("No epoch samples found. Run training first.")
else:
    indices  = np.linspace(0, len(epoch_files) - 1,
                           min(5, len(epoch_files)), dtype=int)
    selected = [epoch_files[i] for i in indices]

    fig, axes = plt.subplots(1, len(selected), figsize=(4 * len(selected), 5))
    if len(selected) == 1:
        axes = [axes]

    for ax, fname in zip(axes, selected):
        epoch_num = fname.replace("epoch_", "").replace(".png", "")
        img = Image.open(os.path.join(samples_dir, fname))
        ax.imshow(img)
        ax.set_title(f"Epoch {epoch_num}", fontsize=11)
        ax.axis("off")

    plt.suptitle("Generator Quality Over Training (rows = classes)", fontsize=13)
    plt.tight_layout()
    plt.show()
    print(f"Showing {len(selected)} of {len(epoch_files)} saved epochs")

## Results Summary

| Metric | Value |
|--------|-------|
| Final Critic Loss    | *see output above* |
| Final Generator Loss | *see output above* |
| Final W Distance     | *see output above* |
| Training Epochs      | see config |

**Key Takeaways:**
- **Wasserstein distance** is a more meaningful metric than BCE loss —  
  it measures actual distribution distance, not just classification accuracy  
- **Gradient penalty** replaces weight clipping — softer, more stable Lipschitz enforcement  
- **beta1=0.0** in Adam: no momentum — critical for WGAN-GP stability  
- **Critic × 5**: critic must be accurate before G updates — otherwise  
  G receives misleading gradients from an undertrained critic  
- **Conditioning quality**: each row should look like its class label.  
  Trouser and Bag are easiest. Shirt vs T-shirt vs Pullover are hardest.